# 지도 학습 — 회귀(1) : 선형 회귀 · 평가 지표 · Lasso / Ridge

수업 중 작성한 필기 `Data_Preprocessing02` 와 개념 메모 `notbook02.py` 를 **실행 가능하도록 정리한 노트**입니다.
(교재 `머신러닝` p.29~46 — 10. 지도 학습을 이용한 회귀(1))

메모에는 수업 중 받아 적다 틀린 부분이 있어, **교재와 실행 결과로 확인한 내용만** 남기고 틀린 부분은 고쳐 적었습니다.
원본에서 고친 부분은 **📝 고친 부분** 으로 따로 적어 두었습니다.

> 실습 데이터 `house_price.csv` (보스턴 주택 가격, 506행) 는 이 노트북과 **같은 폴더**에 있어야 합니다.
> `scikit-learn` 이 필요합니다 → `pip install scikit-learn`

## 1. 회귀(Regression)와 선형 회귀 (교재 p.30~33)

**회귀** — 학습 데이터로 특성과 정답의 관계를 파악하고, 학습 데이터에 없는 **미지의 데이터**가 주어졌을 때 **연속적인 숫자 값**을 예측하는 것.
예) 공부 시간 → 시험 성적

**선형 회귀(Linear Regression)** — 입력(x)과 출력(y)의 관계를 **직선**으로 나타내고, 미지의 데이터도 그 직선으로 예측한다.

$$y = Wx + b \qquad (W : \text{가중치(기울기)},\ b : \text{절편})$$

| 용어 | 뜻 | 보스턴 실습에서 |
|------|-----|---------------|
| **특성** x (Feature) | 예측에 쓰는 값 (문제) | 범죄율, 방 개수 등 13개 |
| **정답** y (Label) | 맞혀야 할 값 | `medv` 집값 |
| **모델** | x 를 넣으면 y 를 내놓는 **직선(함수)** | `LinearRegression` |
| **학습**(`fit`) | 데이터를 가장 잘 표현하는 **W, b 찾기** | 학습용 데이터만 사용 |
| **예측**(`predict`) | 찾은 직선에 새 x 를 넣어 y 계산 | 평가용 데이터로 |

선형 회귀가 하는 일은 결국 **"여러 직선 중 어떤 것이 데이터를 가장 잘 표현하는가?"** 를 찾는 것이다. (p.33 "정확한 직선 찾기")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# 교재 p.30~31 : 공부 시간(x) → 시험 성적(y)
X_hour = np.array([[9], [8], [4], [2]])   # 특성은 2차원 (행: 데이터, 열: 특성)
y_score = np.array([90, 80, 40, 20])

model = LinearRegression()
model.fit(X_hour, y_score)                # 학습 : 가장 잘 맞는 W, b 찾기

print("기울기 W :", model.coef_)
print("절편   b :", round(model.intercept_, 2))
print("7시간 공부하면? :", model.predict([[7]]))   # y = 10 * 7 + 0

> 점 4개가 정확히 `y = 10x` 위에 있어서 7시간 → **70점**.
> 실제 데이터(p.32 공부 시간 9 → 74점, 14 → 81점 …)는 점들이 직선 위에 딱 놓이지 않는다. 그래서 **오차**가 생기고, 오차가 가장 작은 직선을 찾는다.

## 2. 선형 회귀 모델의 성능 평가 (교재 p.34~37)

**오차(Error, 손실 loss) = 정답 − 예측** ($y - \hat{y}$). 오차가 **작을수록** 좋은 모델이다.
점마다 오차가 다르니, 여러 오차를 **하나의 숫자로 요약**한 것이 평가 지표다.

| 지표 | 공식 | 뜻 | scikit-learn |
|------|------|-----|--------------|
| **MAE** (평균 절대값 오차) | $\dfrac{\sum \lvert y - \hat{y} \rvert}{n}$ | 오차의 **절댓값** 평균 | `mean_absolute_error` |
| **MSE** (평균 제곱 오차) | $\dfrac{\sum (y - \hat{y})^2}{n}$ | 오차의 **제곱** 평균 — 큰 오차를 더 크게 벌한다 | `mean_squared_error` |
| **RMSE** (평균 제곱근 오차) | $\sqrt{\dfrac{\sum (y - \hat{y})^2}{n}}$ | MSE 에 **루트** — 단위가 정답과 같아진다 | `root_mean_squared_error` |

> 왜 절댓값이나 제곱을 할까? 오차가 +3, −3 이면 그냥 더했을 때 0 이 되어 "오차가 없다"로 보이기 때문이다.
> MSE 는 제곱 때문에 단위도 **제곱**이 된다(집값 단위가 천 달러면 MSE 는 "천 달러²"). 루트를 씌운 RMSE 는 **"평균적으로 몇 천 달러 틀렸다"** 처럼 바로 읽을 수 있다.

In [ ]:
# 작은 예시로 세 지표를 직접 계산해 보기
y_true = np.array([3, -0.5, 2, 7])     # 정답
y_hat  = np.array([2.5, 0.0, 2, 8])    # 예측

error = y_true - y_hat                  # 오차 = 정답 - 예측
print("오차 :", error)

mae  = np.mean(np.abs(error))           # 절댓값의 평균
mse  = np.mean(error ** 2)              # 제곱의 평균
rmse = np.sqrt(mse)                     # MSE 에 루트
print(f"MAE = {mae:.3f}, MSE = {mse:.3f}, RMSE = {rmse:.3f}")

> **📝 고친 부분** (`notbook02.py` 메모)
> - "정답 − 예측" → 맞다. 오차 $= y - \hat{y}$
> - "정답 ± 평균 오차를 계산 후 (예측 데이터)" → 예측값은 **학습한 직선** $\hat{y} = Wx + b$ 로 구한다. 평균 오차는 예측이 얼마나 틀렸는지 재는 **평가 지표**다.
> - "평균제곱 오차 숫자가 너무 크기 때문에 (RMSE)" → 반만 맞다. 핵심은 **제곱 때문에 단위가 바뀐 것**을 루트로 **원래 단위로 되돌리는** 것
> - "평균제곱 오차 **MASS** 에 루트를 씌운 것" → **MSE**
> - 필기 첫 셀 "**REMSE** 학습한다" → **RMSE**

## 3. 보스턴 주택 가격 데이터 (교재 p.39)

집값에 영향을 주는 **특성 13가지, 506개** 지역 데이터. 정답은 `medv` (주택 가격 중앙값, 단위 천 달러).

| 컬럼 | 뜻 | 컬럼 | 뜻 |
|------|-----|------|-----|
| `crim` | 인구 1인당 범죄 발생 수 | `dis` | 보스턴 고용 시설 5곳까지의 거리 |
| `zn` | 25,000평방피트 이상 주거 구역 비중 | `rad` | 순환 고속도로 접근 용이성 |
| `indus` | 소매업 외 상업 면적 비율 | `tax` | $10,000 당 재산세 |
| `chas` | 찰스강 인접 여부 (1: 강 주변) | `ptratio` | 학생 대 교사 비율 |
| `nox` | 일산화질소 농도 | `b` | 지역별 흑인 비율 관련 지표 |
| `rm` | 집의 평균 방 개수 | `lstat` | 저소득층 인구 비율(%) |
| `age` | 1940년 이전에 지어진 집 비율 | **`medv`** | **주택 가격 (정답)** |

> 이 데이터는 `b` 컬럼이 인종을 변수로 쓴다는 윤리 문제로 scikit-learn 1.2 부터 내장 데이터에서 빠졌다. 지금은 **회귀 실습용**으로만 쓴다.

In [ ]:
# 데이터 불러오기 (노트북과 같은 폴더)
path = "./house_price.csv"
df = pd.read_csv(path)

print("모양 :", df.shape)
df.head()

In [ ]:
# 정답(medv)과 각 특성의 상관계수 — 어떤 특성이 집값과 같이 움직일까?
print(df.corr()['medv'].drop('medv').sort_values().round(2))

> 방 개수 `rm` 이 많을수록 비싸고(**+0.70**), 저소득층 비율 `lstat` 이 높을수록 싸다(**−0.74**).

## 4. 데이터 나누기 — 특성/정답 → 학습용/평가용

나누기는 **두 번** 한다.

| 단계 | 나누는 기준 | 결과 |
|------|------------|------|
| ① **열** 나누기 | 문제(특성) vs 정답 | `X_data` (13열) · `y_data` (`medv`) |
| ② **행** 나누기 | 학습용 vs 평가용 | `X_train` · `X_test` · `y_train` · `y_test` (4분할) |

In [ ]:
# ① 열 나누기 : 특성(X) 과 정답(y)
X_data = df.drop(columns=['medv'])     # 특성 13개 (문제)
y_data = df['medv']                    # 정답 (집값)

print("X_data :", X_data.shape, "/ y_data :", y_data.shape)

In [ ]:
from sklearn.model_selection import train_test_split

# ② 행 나누기 : 학습용 75% / 평가용 25% → X, y 각각 둘씩 4분할
X_train, X_test, y_train, y_test = train_test_split(
    X_data,
    y_data,
    test_size=0.25,     # 평가용 비율 25%
    random_state=0      # 섞는 방식 고정 → 실행할 때마다 똑같이 나뉜다 (결과 재현)
)

print("학습용 :", X_train.shape, y_train.shape)
print("평가용 :", X_test.shape, y_test.shape)

print("=" * 40)

# X_train 과 y_train 의 행 번호가 같은지 확인 → 문제와 정답이 짝을 유지한 채 섞였다
print(X_train.index[:10].tolist())
print(y_train.index[:10].tolist())
print("짝이 맞는가? :", (X_train.index == y_train.index).all())

> **📝 고친 부분**
> - `X_data` 에 "#**학습용**", `y_data` 에 "#**평가용**" 주석 → 틀렸다. 이 단계는 **특성(문제) / 정답** 나누기이고, 학습용/평가용은 다음 `train_test_split` 에서 나뉜다.
> - `random_state=0 # 0번째 값 출력` → 0번째 값을 뽑는 게 아니라 **무작위로 섞는 방식(시드)을 고정**하는 값이다. 어떤 숫자든 같은 숫자를 주면 같은 결과가 나온다.
> - `print(y_train.index)  # 평가용(정답)` → `y_train` 은 **학습용 정답**. 평가용 정답은 `y_test`
> - 오타 `X_trarin` → `X_train`
> - 절대 경로 `C:\Users\Jinho\work\house_price.csv` → 상대 경로 `./house_price.csv`

## 5. 표준화 — 학습용으로 기준을 잡고, 평가용에도 같은 기준 적용

| 데이터 | 함수 | 하는 일 |
|--------|------|---------|
| **학습용** | `fit_transform()` | 학습용의 평균·표준편차를 **계산(fit)** 하고 변환 |
| **평가용** | `transform()` | **학습용의** 평균·표준편차로 변환만 (새로 계산 ✕) |

> 왜 평가용으로 `fit` 하면 안 될까? 평가용은 "아직 본 적 없는 미래 데이터" 역할이다.
> 평가용의 평균을 미리 계산해 쓰면 **답안지를 살짝 엿본 것**(데이터 누수)이 되어 성능이 부풀려진다.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()    # 표준화 도구 생성

X_train_scal = scaler.fit_transform(X_train)   # 학습용 : 기준 계산 + 변환
X_test_scal  = scaler.transform(X_test)        # 평가용 : 학습용 기준으로 변환만

print("학습용 평균(≈0) :", X_train_scal.mean(axis=0).round(2)[:5], "...")
print("평가용 평균      :", X_test_scal.mean(axis=0).round(2)[:5], "...  ← 정확히 0 은 아니다 (기준이 학습용이라)")

> **📝 고친 부분**
> - 메모 "학습할 때**만** 표준화를 적용한다" → 틀렸다. **기준 계산(fit)** 은 학습용에서만 하지만, 표준화 **적용(transform)** 은 학습용·평가용 **둘 다** 한다. 평가용을 표준화하지 않으면 모델이 엉뚱한 범위의 값을 받는다.
> - 변수 이름 `Scaler` (대문자, 클래스처럼 보임) → `scaler`
> - 필기의 이 셀 아래 남아 있던 `Index([245, 59, ...])` 출력은 앞 셀 결과가 잘못 붙은 것이라 지웠다
> - 오타 `편균` → 평균

## 6. 선형 회귀 모델 — `LinearRegression` (교재 p.38, p.40~41)

**학습 데이터의 오차(MSE)가 최소가 되는 직선**을 구해 값을 예측하는 모델.
(MSE 가 최소인 직선 = RMSE 가 최소인 직선 — 루트는 크기 순서를 바꾸지 않는다)

```python
from sklearn.linear_model import LinearRegression

lireg = LinearRegression()        # ① 모델 생성
lireg.fit(X_train, y_train)       # ② 학습 : 학습용 데이터로 W, b 찾기
y_pred = lireg.predict(X_test)    # ③ 예측 : 평가용 데이터
```

### 다중 선형 회귀 (p.41)

특성이 **여러 개**면 특성마다 가중치가 하나씩 생긴다.

$$y = w_1x_1 + w_2x_2 + w_3x_3 + \cdots + w_px_p + b$$

보스턴 데이터는 특성이 13개 → 가중치 13개 + 절편 1개. 단순 선형 회귀($y = wx + b$)보다 학습에 쓰는 특성이 많고 복잡하다.

In [ ]:
# ① 모델 생성
lireg = LinearRegression()

# ② 학습 : 학습용(표준화) 데이터로 오차가 최소인 직선 찾기
lireg.fit(X_train_scal, y_train)    # 결과를 따로 변수에 담지 않아도 lireg 자체가 학습된다
print(lireg)

In [ ]:
# 학습 결과 : 특성별 가중치(W, coef_)와 절편(b, intercept_)
coef = pd.Series(lireg.coef_, index=X_data.columns)
print("가중치 W (특성별) :")
print(coef.round(2).sort_values())

print("=" * 40)
print(f"절편 b : {lireg.intercept_:.2f}")

> **가중치 읽는 법** — 표준화한 데이터라 가중치 크기끼리 비교할 수 있다.
> `lstat` **−3.59**, `dis` −3.00 이 가장 크게 집값을 끌어내리고, `rm` **+2.61** 이 가장 크게 끌어올린다.
> 절편 22.61 은 모든 특성이 **평균값**(표준화 후 0)일 때의 예측 집값 = 학습용 집값의 평균이다.

In [ ]:
# ③ 예측 : 평가용 데이터의 집값
y_pred = lireg.predict(X_test_scal)

# 정답과 예측을 나란히 비교
compare = pd.DataFrame({'정답': y_test.values, '예측': y_pred.round(1)}, index=y_test.index)
compare['오차'] = (compare['정답'] - compare['예측']).round(1)
compare.head(10)

### 모델 평가 — MAE · MSE · RMSE

In [ ]:
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import root_mean_squared_error as rmse   # scikit-learn 1.4 이상

test_mae  = mae(y_test, y_pred)
test_mse  = mse(y_test, y_pred)
test_rmse = rmse(y_test, y_pred)

print(f"MAE  : {test_mae:.2f}")
print(f"MSE  : {test_mse:.2f}")
print(f"RMSE : {test_rmse:.2f}   (= √MSE = {np.sqrt(test_mse):.2f})")

> **결과 해석** — RMSE **5.46** → 평가용 집값을 평균적으로 **약 5,460달러** 정도 틀리게 예측했다. (집값 평균은 약 22,500달러)

> **📝 고친 부분**
> - `train = lireg.fit(...)` + "lireg 변수에 대입하지 않으면 (에러 원인)" → 틀렸다. `fit()` 은 **`lireg` 자기 자신을 학습시키고** 자기 자신을 돌려준다. 그래서 `train` 과 `lireg` 는 **같은 객체**이고, 대입하지 않아도 에러가 나지 않는다.
> - `coef_` 를 "학습용 최소 **기울기**", `intercept_` 를 "모델 **절연**" 으로 적은 것 → `coef_` 는 특성별 **가중치 W**(13개), `intercept_` 는 **절편 b**
> - "X(컬럼) 선이 하나면 Y = wx + b" → 특성이 13개라 **다중 선형 회귀** $y = w_1x_1 + \cdots + w_{13}x_{13} + b$
> - 메모 "MSE : 학습에 최소물" → 학습 = **학습 데이터의 MSE 가 최소**가 되는 W, b 를 찾는 것
> - 변수 이름 `prad` → `y_pred`, 출력 `mean_mse29.78` → `MSE : 29.78` (콜론 빠짐)
> - 셀 끝의 `display(y_test)` → 정답·예측·오차를 나란히 놓은 표로
> - 교재 p.35 **MAE** 추가

In [ ]:
# 참고 : 표준화를 안 하면 선형 회귀 결과가 달라질까?
lireg_raw = LinearRegression().fit(X_train, y_train)
print(f"표준화 X  RMSE : {rmse(y_test, lireg_raw.predict(X_test)):.2f}")
print(f"표준화 O  RMSE : {test_rmse:.2f}")

> **같다.** 기본 선형 회귀는 표준화해도 **가중치 크기만 바뀌고 예측은 똑같다.**
> 그런데도 표준화하는 이유 — ① 가중치끼리 **크기 비교**가 가능해지고, ② 다음 단원의 **Lasso / Ridge** 처럼 가중치 크기에 벌점을 주는 모델은 표준화하지 않으면 결과가 달라진다. (Day 020 스케일링 가이드 : 선형 & 규제 모델 → `StandardScaler`)

## 7. 모델 정규화(Regularization) — 규제 (교재 p.42~44)

**모델이 과대적합되지 않도록, 복잡한 모델을 단순하게 만드는 것.**

- **과대적합(overfitting)** — 학습 데이터에만 너무 딱 맞춰져서, 처음 보는 데이터에는 오히려 틀리는 것 (학습용 점수 ≫ 평가용 점수)
- 특성(데이터)은 그대로 두고 복잡도를 줄이려면 → **가중치 W 를 0 이나 0 에 가까운 작은 숫자로** 만든다

> ⚠️ 이름 주의 — 여기서 **정규화(Regularization) = 규제**다. Day 020 의 스케일링 **정규화(Normalization, MinMaxScaler)** 와 **전혀 다른 개념**이다.

| | **L1 규제** → `Lasso` | **L2 규제** → `Ridge` |
|---|---|---|
| 줄이는 방식 | 모든 W 를 **같은 크기**만큼 감소 ($w - k$) | 모든 W 를 **같은 비율**로 감소 (0.9 → 0.45, 0.2 → 0.1) |
| 효과 | 작은 가중치는 **0** 이 된다 → 중요도 낮은 특성을 **제거** | 큰 가중치일수록 더 크게 줄어 전체가 0 에 **가까워진다** (0 은 잘 안 됨) |
| 교재 예 | 국·영·수 0.9, 체육·음악·미술 0.2 → −0.2 → 국영수 0.7 만 남음 | −50% → 국영수 0.45, 체육·음악·미술 0.1 |

**하이퍼파라미터 `alpha` (α)** — 규제의 **강도**. 클수록 가중치를 세게 줄인다 (기본값 1.0).

## 8. Lasso · Ridge 모델 (교재 p.45~46)

```python
from sklearn.linear_model import Lasso, Ridge

lasso = Lasso(alpha=1.0, random_state=0)    # 모델 생성 + 하이퍼파라미터 설정
lasso.fit(X_train, y_train)                 # 학습
lasso.predict(X_test)                       # 예측

ridge = Ridge(alpha=1.0, random_state=0)    # 사용법은 LinearRegression 과 같다
```

> 교재에는 `Lasso(random_state)` 로만 적혀 있지만, 규제의 핵심 설정값은 **`alpha`** 다.
> (`random_state` 는 내부 계산 순서를 무작위로 할 때만 쓰여서, 기본 설정에서는 결과에 영향이 없다)

In [ ]:
from sklearn.linear_model import Lasso, Ridge

# 같은 학습용(표준화) 데이터로 세 모델 학습
lasso = Lasso(alpha=1.0, random_state=0)
ridge = Ridge(alpha=1.0, random_state=0)
lasso.fit(X_train_scal, y_train)
ridge.fit(X_train_scal, y_train)

# 특성별 가중치 비교
coef_table = pd.DataFrame({
    'LinearRegression': lireg.coef_,
    'Ridge (L2)': ridge.coef_,
    'Lasso (L1)': lasso.coef_,
}, index=X_data.columns).round(2)
coef_table

In [ ]:
# 가중치가 정확히 0 이 된 특성 개수
for name, m in [('LinearRegression', lireg), ('Ridge', ridge), ('Lasso', lasso)]:
    zero = (m.coef_ == 0).sum()
    print(f"{name:16s} : 0 이 된 가중치 {zero:2d}개, 평가용 RMSE {rmse(y_test, m.predict(X_test_scal)):.2f}")

print("=" * 60)
print("Lasso 가 남긴 특성 :", X_data.columns[lasso.coef_ != 0].tolist())

> **결과 해석**
> - **Lasso** 는 13개 중 **8개** 가중치를 0 으로 만들고 `rm` · `tax` · `ptratio` · `b` · `lstat` **5개만 남겼다** → 특성 선택 효과
> - **Ridge** 는 0 이 된 가중치가 없고, 전체 크기만 조금씩 줄었다
> - 평가용 RMSE 는 선형 회귀 · Ridge 가 **5.46** 으로 거의 같고, 특성을 8개나 버린 Lasso 는 **5.96** 으로 조금 나빠졌다.
>   이 데이터는 과대적합이 심하지 않아서다. 규제는 특성이 많고 데이터가 적어 **과대적합이 생길 때** 효과가 크다.

In [ ]:
# alpha (규제 강도) 를 바꿔 가며 비교
rows = []
for alpha in [0.01, 0.1, 1, 10]:
    for name, Model in [('Lasso', Lasso), ('Ridge', Ridge)]:
        m = Model(alpha=alpha, random_state=0).fit(X_train_scal, y_train)
        rows.append({
            '모델': name,
            'alpha': alpha,
            '0 인 가중치 수': int((m.coef_ == 0).sum()),
            '가중치 크기 합': round(np.abs(m.coef_).sum(), 2),
            '평가용 RMSE': round(rmse(y_test, m.predict(X_test_scal)), 2),
        })

pd.DataFrame(rows)

> - `alpha` 가 커질수록 가중치 크기 합이 줄어든다 = **규제가 세진다**
> - Lasso 는 `alpha=10` 에서 13개 가중치가 **전부 0** → 모든 집을 평균값으로만 예측하는 모델이 되어 RMSE 가 9.04 로 크게 나빠졌다. 규제가 **너무 세면 과소적합**
> - Ridge 는 같은 `alpha` 에서도 훨씬 완만하게 줄어든다 — 적당한 `alpha` 를 고르는 것이 하이퍼파라미터 튜닝이다

## 9. (보류) 로지스틱 회귀 — 다음 시간 예고

필기 마지막의 "(보류)" 메모는 아직 교재 범위(p.46) 밖이라, 확인된 내용만 짧게 정리해 둔다.

- **로지스틱 회귀**는 이름은 회귀지만 **분류** 모델이다. (합격/불합격처럼 범주를 예측)
- 직선 $z = Wx + b$ 의 값을 **시그모이드 함수**에 넣어 **0 ~ 1 사이의 확률**로 바꾼다.

$$\sigma(z) = \frac{1}{1 + e^{-z}} \qquad (e \approx 2.718,\ \text{자연 상수})$$

- 확률이 **0.5 이상**이면 1(예: 합격), 미만이면 0 → 이 기준이 되는 선이 **결정 경계**

> **📝 고친 부분** — 메모 "Y = X + 1", "X^+2 (3^2 = 9)", "+,−50% 라서 어디라고 말하기 애매하다" 는 받아 적다 섞인 것으로 보여 뺐다. 시그모이드에 쓰이는 "딱 떨어지지 않는 수 e(자연 상수)" 는 맞는 내용이라 남겼다.

## 10. 오늘 정리

| 단계 | 코드 | 핵심 |
|------|------|------|
| 특성/정답 나누기 | `X = df.drop(columns=['medv'])`, `y = df['medv']` | 열 기준 |
| 학습/평가 나누기 | `train_test_split(X, y, test_size=0.25, random_state=0)` | 행 기준, 4분할 |
| 표준화 | `fit_transform(X_train)` / `transform(X_test)` | 기준은 **학습용**에서만 |
| 학습 | `LinearRegression().fit(X_train, y_train)` | MSE 최소인 W, b |
| 예측 | `model.predict(X_test)` | |
| 평가 | `mae` · `mse` · `rmse(y_test, y_pred)` | 작을수록 좋다 |
| 규제 | `Lasso(alpha)` (L1) · `Ridge(alpha)` (L2) | 가중치를 줄여 과대적합 방지 |

- 선형 회귀 = 데이터를 가장 잘 표현하는 **직선** $y = Wx + b$ 찾기. 특성이 여러 개면 **다중 선형 회귀**.
- `coef_` = 특성별 **가중치 W**, `intercept_` = **절편 b**.
- **MAE** 절댓값 평균 · **MSE** 제곱 평균 · **RMSE** = √MSE (정답과 같은 단위).
- **L1(Lasso)** 은 가중치를 0 으로 만들어 특성을 **골라내고**, **L2(Ridge)** 는 전체를 **고르게 줄인다**. 강도는 `alpha`.
- 규제의 "정규화(Regularization)" ≠ 스케일링의 "정규화(Normalization)".